In [1]:
from pathlib import Path
import json
import sys
import re


import numpy as np
import pandas as pd

from sklearn.inspection import permutation_importance

from joblib import Parallel, delayed
from collections import Counter
import os
import time


project_dir = Path.cwd()

raw_dir = project_dir / "data" / "raw"
output_dir = project_dir / "outputs" / "poisson"

results_path = raw_dir / "results.csv"
rankings_path = raw_dir / "fifa_consolidado.csv"
schedule_path = raw_dir / "world_cup_2026_schedule.csv"
odds_json_path = raw_dir / "odds_fase_grupos_copa.json"
odds_csv_path = raw_dir / "odds.csv"

output_dir.mkdir(parents=True, exist_ok=True)

print(project_dir)
print(raw_dir.exists())

d:\windows\Documents\extras\previsao_copa_26
True


In [2]:
TEAM_NAME_ALIASES = {
    # América do Norte / Caribe
    "USA": "United States",
    "United States of America": "United States",
    "US Virgin Islands": "United States Virgin Islands",
    "St Kitts and Nevis": "Saint Kitts and Nevis",
    "St. Kitts and Nevis": "Saint Kitts and Nevis",
    "St Lucia": "Saint Lucia",
    "St. Lucia": "Saint Lucia",
    "St Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "St. Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "St. Vincent / Grenadines": "Saint Vincent and the Grenadines",
    "Curacao": "Curacao",
    "Curaçao": "Curacao",

    # África
    "Cape Verde": "Cabo Verde",
    "Cape Verde Islands": "Cabo Verde",
    "Cabo Verde": "Cabo Verde",
    "DR Congo": "Congo DR",
    "Democratic Republic of Congo": "Congo DR",
    "Congo DR": "Congo DR",
    "Ivory Coast": "Ivory Coast",
    "Côte d'Ivoire": "Ivory Coast",
    "The Gambia": "Gambia",
    "São Tomé and Príncipe": "Sao Tome e Principe",
    "São Tomé e Príncipe": "Sao Tome e Principe",
    "Sao Tome e Principe": "Sao Tome e Principe",

    # Ásia
    "China PR": "China",
    "Hong Kong, China": "Hong Kong",
    "Korea Republic": "South Korea",
    "Korea DPR": "North Korea",
    "IR Iran": "Iran",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Brunei Darussalam": "Brunei",

    # Europa
    "Czech Republic": "Czechia",
    "Türkiye": "Turkey",
    "Republic of Ireland": "Ireland",
    "FYR Macedonia": "North Macedonia",
    "Serbia and Montenegro" : "Serbia",

    # Oceania
    "Aotearoa New Zealand": "New Zealand",

    # Casos com marcador "(unranked)"
    "American Samoa (unranked)": "American Samoa",
    "Eritrea (unranked)": "Eritrea",
    "Samoa (unranked)": "Samoa",
    "Tonga (unranked)": "Tonga",
}

In [3]:
def normalize_team_name_extended(team_name):
    if pd.isna(team_name):
        return team_name

    team_name = str(team_name).strip()

    # Remove notas de Wikipedia: [nb 3], [1], etc.
    team_name = re.sub(r"\[.*?\]", "", team_name).strip()

    # Remove marcadores comuns
    team_name = team_name.replace("(H)", "").strip()
    team_name = team_name.replace("(unranked)", "").strip()

    # Ajusta espaços duplicados
    team_name = re.sub(r"\s+", " ", team_name).strip()

    # Aplica alias explícito
    team_name = TEAM_NAME_ALIASES.get(team_name, team_name)

    return team_name    

In [4]:
TEAM_NAME_MAP = {
    "USA": "United States",
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "Curaçao": "Curacao",
    "Côte d'Ivoire": "Ivory Coast",
    "Czech Republic": "Czechia",
    "Türkiye": "Turkey",
}


def clean_team_name(team_name):
    team_name = str(team_name).strip()
    return TEAM_NAME_MAP.get(team_name, team_name)


def extract_h2h_prices(event):
    home_team = clean_team_name(event["home_team"])
    away_team = clean_team_name(event["away_team"])

    home_prices = []
    draw_prices = []
    away_prices = []

    for bookmaker in event.get("bookmakers", []):
        for market in bookmaker.get("markets", []):
            if market.get("key") != "h2h":
                continue

            prices_by_name = {
                clean_team_name(outcome["name"]): outcome["price"]
                for outcome in market.get("outcomes", [])
            }

            if home_team in prices_by_name:
                home_prices.append(float(prices_by_name[home_team]))

            if away_team in prices_by_name:
                away_prices.append(float(prices_by_name[away_team]))

            if "Draw" in prices_by_name:
                draw_prices.append(float(prices_by_name["Draw"]))

    if not home_prices or not draw_prices or not away_prices:
        return None

    return {
        "event_date": pd.to_datetime(event["commence_time"]).date(),
        "event_home_team": home_team,
        "event_away_team": away_team,
        "event_home_win_odds": float(np.median(home_prices)),
        "event_draw_odds": float(np.median(draw_prices)),
        "event_away_win_odds": float(np.median(away_prices)),
        "n_bookmakers": len(home_prices),
    }


with odds_json_path.open("r", encoding="utf-8") as file:
    odds_events = json.load(file)

odds_rows = []

for event in odds_events:
    extracted_prices = extract_h2h_prices(event)

    if extracted_prices is not None:
        odds_rows.append(extracted_prices)

odds_data = pd.DataFrame(odds_rows)

print(odds_data.shape)
odds_data.head()

(72, 7)


,event_date,event_home_team,event_away_team,event_home_win_odds,event_draw_odds,event_away_win_odds,n_bookmakers
0,2026-06-11,Mexico,South Africa,1.41,4.50,8.725,28
1,2026-06-12,South Korea,Czechia,2.70,3.10,2.850,27
2,2026-06-12,Canada,Bosnia and Herzegovina,1.81,3.60,4.700,27
3,2026-06-13,United States,Paraguay,1.94,3.40,4.200,27
4,2026-06-13,Qatar,Switzerland,13.25,6.75,1.200,27


In [5]:
schedule_data = pd.read_csv(schedule_path)

schedule_data["home_team"] = schedule_data["home_team"].map(clean_team_name)
schedule_data["away_team"] = schedule_data["away_team"].map(clean_team_name)

aligned_rows = []

for _, odds_row in odds_data.iterrows():
    event_home_team = odds_row["event_home_team"]
    event_away_team = odds_row["event_away_team"]

    same_order = schedule_data[
        (schedule_data["home_team"] == event_home_team)
        & (schedule_data["away_team"] == event_away_team)
    ]

    reverse_order = schedule_data[
        (schedule_data["home_team"] == event_away_team)
        & (schedule_data["away_team"] == event_home_team)
    ]

    if not same_order.empty:
        schedule_row = same_order.iloc[0]

        aligned_rows.append(
            {
                "match_date": schedule_row["match_date"],
                "home_team": schedule_row["home_team"],
                "away_team": schedule_row["away_team"],
                "home_win_odds": odds_row["event_home_win_odds"],
                "draw_odds": odds_row["event_draw_odds"],
                "away_win_odds": odds_row["event_away_win_odds"],
                "n_bookmakers": odds_row["n_bookmakers"],
            }
        )

    elif not reverse_order.empty:
        schedule_row = reverse_order.iloc[0]

        aligned_rows.append(
            {
                "match_date": schedule_row["match_date"],
                "home_team": schedule_row["home_team"],
                "away_team": schedule_row["away_team"],
                "home_win_odds": odds_row["event_away_win_odds"],
                "draw_odds": odds_row["event_draw_odds"],
                "away_win_odds": odds_row["event_home_win_odds"],
                "n_bookmakers": odds_row["n_bookmakers"],
            }
        )

    else:
        print("Não encontrei na tabela:", event_home_team, "x", event_away_team)

odds_aligned_data = pd.DataFrame(aligned_rows)

odds_aligned_data.to_csv(odds_csv_path, index=False)

print(odds_aligned_data.shape)
odds_aligned_data.head()

(72, 7)


,match_date,home_team,away_team,home_win_odds,draw_odds,away_win_odds,n_bookmakers
0,2026-06-11,Mexico,South Africa,1.41,4.50,8.725,28
1,2026-06-11,South Korea,Czechia,2.70,3.10,2.850,27
2,2026-06-12,Canada,Bosnia and Herzegovina,1.81,3.60,4.700,27
3,2026-06-12,United States,Paraguay,1.94,3.40,4.200,27
4,2026-06-13,Qatar,Switzerland,13.25,6.75,1.200,27


In [6]:
import poison_26 as path_a

In [7]:
historical_matches_raw = pd.read_csv(results_path)
rankings_raw = pd.read_csv(rankings_path)
schedule_raw = pd.read_csv(schedule_path)
odds_raw = pd.read_csv(odds_csv_path)

historical_matches_raw = historical_matches_raw.copy()

score_columns = ["home_score", "away_score"]

# Histórico de jogos
for column in ["home_team", "away_team"]:
    if column in historical_matches_raw.columns:
        historical_matches_raw[column] = historical_matches_raw[column].map(
            normalize_team_name_extended
        )

# Ranking FIFA
possible_ranking_team_columns = [
    "team",
    "country_full",
    "country",
    "name",
    "nation",
]

for column in possible_ranking_team_columns:
    if column in rankings_raw.columns:
        rankings_raw[column] = rankings_raw[column].map(
            normalize_team_name_extended
        )

# Tabela da Copa
for column in ["home_team", "away_team"]:
    if column in schedule_raw.columns:
        schedule_raw[column] = schedule_raw[column].map(
            normalize_team_name_extended
        )

# Odds, se estiver usando
if odds_csv_path.exists():
    odds_raw = pd.read_csv(odds_csv_path)

    for column in ["home_team", "away_team"]:
        if column in odds_raw.columns:
            odds_raw[column] = odds_raw[column].map(
                normalize_team_name_extended
            )

    odds_raw.to_csv(odds_csv_path, index=False)

historical_matches_raw["date"] = pd.to_datetime(
    historical_matches_raw["date"],
    errors="coerce",
)

historical_matches_raw["home_score"] = pd.to_numeric(
    historical_matches_raw["home_score"],
    errors="coerce",
)

historical_matches_raw["away_score"] = pd.to_numeric(
    historical_matches_raw["away_score"],
    errors="coerce",
)

missing_score_data = historical_matches_raw[
    historical_matches_raw[score_columns].isna().any(axis=1)
].copy()

print("Jogos sem placar:", missing_score_data.shape[0])

display(
    missing_score_data[
        ["date", "home_team", "away_team", "home_score", "away_score", "tournament"]
    ].tail(20)
)

historical_matches_clean = historical_matches_raw.dropna(
    subset=["date", "home_score", "away_score"]
).copy()

historical_matches_clean = historical_matches_clean[
    historical_matches_clean["date"] < pd.Timestamp("2026-06-11")
].copy()

historical_matches_clean["home_score"] = historical_matches_clean["home_score"].astype(int)
historical_matches_clean["away_score"] = historical_matches_clean["away_score"].astype(int)

historical_matches = path_a.standardize_matches_columns(historical_matches_clean)
schedule_data = path_a.standardize_schedule_columns(schedule_raw)

ranking_lookup = path_a.RankingLookup(rankings_raw)
odds_lookup = path_a.OddsLookup(odds_raw)

print("Histórico bruto:", historical_matches_raw.shape)
print("Histórico limpo:", historical_matches.shape)
print("Tabela Copa:", schedule_data.shape)
print("Odds:", odds_raw.shape)

Jogos sem placar: 72


,date,home_team,away_team,home_score,away_score,tournament
49452,2026-06-24,Scotland,Brazil,NaN,NaN,FIFA World Cup
49453,2026-06-24,Morocco,Haiti,NaN,NaN,FIFA World Cup
49454,2026-06-25,United States,Turkey,NaN,NaN,FIFA World Cup
49455,2026-06-25,Paraguay,Australia,NaN,NaN,FIFA World Cup
49456,2026-06-25,Curacao,Ivory Coast,NaN,NaN,FIFA World Cup
49457,2026-06-25,Ecuador,Germany,NaN,NaN,FIFA World Cup
49458,2026-06-25,Japan,Sweden,NaN,NaN,FIFA World Cup
49459,2026-06-25,Tunisia,Netherlands,NaN,NaN,FIFA World Cup
49460,2026-06-26,Egypt,Iran,NaN,NaN,FIFA World Cup
49461,2026-06-26,New Zealand,Belgium,NaN,NaN,FIFA World Cup


Histórico bruto: (49472, 9)
Histórico limpo: (49400, 9)
Tabela Copa: (72, 8)
Odds: (72, 7)


In [8]:
market_features = [
    "market_home_prob",
    "market_draw_prob",
    "market_away_prob",
]

path_a.FEATURE_COLUMNS = [
    feature
    for feature in path_a.FEATURE_COLUMNS
    if feature not in market_features
]

odds_lookup_for_training = path_a.OddsLookup(None)

features_data, home_goals, away_goals, initial_states = path_a.build_training_data(
    historical_matches=historical_matches,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup_for_training,
)

home_model = path_a.train_poisson_model(features_data, home_goals)
away_model = path_a.train_poisson_model(features_data, away_goals)

print(features_data.shape)
print(path_a.FEATURE_COLUMNS)

(49400, 23)
['elo_diff', 'attack_diff', 'defense_diff', 'form_diff', 'home_rank', 'away_rank', 'rank_diff', 'home_total_points', 'away_total_points', 'fifa_points_diff', 'neutral', 'is_non_neutral', 'tournament_weight', 'home_attack', 'away_attack', 'home_defense', 'away_defense', 'home_form', 'away_form', 'home_decay', 'away_decay', 'home_matches_played', 'away_matches_played']


In [9]:
features_data, home_goals, away_goals, initial_states = path_a.build_training_data(
    historical_matches=historical_matches,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
)

home_model = path_a.train_poisson_model(features_data, home_goals)
away_model = path_a.train_poisson_model(features_data, away_goals)

print(home_goals.shape)
print(away_goals.shape)

(49400,)
(49400,)


In [10]:
def normalize_team_name_safe(team):
    if hasattr(path_a, "normalize_team_name"):
        return path_a.normalize_team_name(team)

    return str(team).strip()

def extract_teams_from_rankings(rankings_raw):
    possible_team_columns = [
        "team",
        "country_full",
        "country",
        "name",
        "nation",
    ]

    for column in possible_team_columns:
        if column in rankings_raw.columns:
            return set(
                rankings_raw[column]
                .dropna()
                .map(normalize_team_name_safe)
                .unique()
            )

    return set()

def get_all_available_teams(
    initial_states,
    historical_matches,
    rankings_raw,
):
    teams = set()

    teams.update(initial_states.keys())

    if "home_team" in historical_matches.columns:
        teams.update(historical_matches["home_team"].dropna().map(normalize_team_name_safe))

    if "away_team" in historical_matches.columns:
        teams.update(historical_matches["away_team"].dropna().map(normalize_team_name_safe))

    teams.update(extract_teams_from_rankings(rankings_raw))

    teams = sorted(
        team
        for team in teams
        if isinstance(team, str)
        and team.strip() != ""
        and team.lower() != "nan"
    )

    return teams

all_teams_after_normalization = get_all_available_teams(
    initial_states=initial_states,
    historical_matches=historical_matches,
    rankings_raw=rankings_raw,
)

suspicious_terms = [
    "Czech",
    "Congo",
    "Cape",
    "Cabo",
    "Cura",
    "Korea",
    "Iran",
    "Ireland",
    "Macedonia",
    "Turkey",
    "Türkiye",
    "Saint",
    "St ",
    "Tome",
    "Príncipe",
    "Principe",
    "United States",
    "USA",
]

for term in suspicious_terms:
    matches = [
        team for team in all_teams_after_normalization
        if term.lower() in team.lower()
    ]

    if matches:
        print("\n", term)
        print(matches)

all_teams = get_all_available_teams(
    initial_states=initial_states,
    historical_matches=historical_matches,
    rankings_raw=rankings_raw,
)

print("Total de seleções encontradas:", len(all_teams))
print(all_teams[:30])


 Czech
['Czechia', 'Czechoslovakia']

 Congo
['Congo', 'Congo DR']

 Cabo
['Cabo Verde']

 Cura
['Curacao']

 Korea
['DPR Korea', 'North Korea', 'South Korea', 'United Koreans in Japan']

 Iran
['Iran']

 Ireland
['Ireland', 'Northern Ireland']

 Macedonia
['North Macedonia']

 Turkey
['Turkey']

 Saint
['Saint Barthélemy', 'Saint Helena', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Martin', 'Saint Pierre and Miquelon', 'Saint Vincent and the Grenadines']

 St 
['East Turkestan', 'West Papua']

 Tome
['Sao Tome e Principe']

 Principe
['Sao Tome e Principe']

 United States
['United States', 'United States Virgin Islands']
Total de seleções encontradas: 341
['Abkhazia', 'Afghanistan', 'Albania', 'Alderney', 'Algeria', 'Ambazonia', 'American Samoa', 'Andalusia', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Arameans Suryoye', 'Argentina', 'Armenia', 'Artsakh', 'Aruba', 'Asturias', 'Australia', 'Austria', 'Aymara', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barawa'

In [11]:
def extract_poisson_weights(model, model_name):
    scaler = model.named_steps["scaler"]
    poisson = model.named_steps["poisson"]

    standardized_coefficients = pd.Series(
        poisson.coef_,
        index=path_a.FEATURE_COLUMNS,
        name="standardized_coefficient",
    )

    raw_scale_coefficients = standardized_coefficients / scaler.scale_

    weights_data = pd.DataFrame(
        {
            "feature": path_a.FEATURE_COLUMNS,
            "standardized_coefficient": standardized_coefficients.values,
            "raw_scale_coefficient": raw_scale_coefficients.values,
            "abs_standardized_coefficient": standardized_coefficients.abs().values,
        }
    )

    weights_data["model"] = model_name

    return weights_data.sort_values(
        "abs_standardized_coefficient",
        ascending=False,
    ).reset_index(drop=True)


home_weights = extract_poisson_weights(home_model, "home_goals")
away_weights = extract_poisson_weights(away_model, "away_goals")

display(home_weights.head(30))
display(away_weights.head(30))

,feature,standardized_coefficient,raw_scale_coefficient,abs_standardized_coefficient,model
0,elo_diff,0.355819,0.001519,0.355819,home_goals
1,away_defense,0.112705,0.151912,0.112705,home_goals
2,away_matches_played,-0.104957,-0.000470,0.104957,home_goals
3,form_diff,-0.104567,-0.499397,0.104567,home_goals
4,home_attack,0.092997,0.154063,0.092997,home_goals
5,defense_diff,0.083969,0.099869,0.083969,home_goals
6,away_form,0.078900,0.494345,0.078900,home_goals
7,attack_diff,0.064220,0.090172,0.064220,home_goals
8,home_form,-0.058116,-0.363102,0.058116,home_goals
9,away_decay,-0.044172,-0.276153,0.044172,home_goals


,feature,standardized_coefficient,raw_scale_coefficient,abs_standardized_coefficient,model
0,elo_diff,-0.359674,-0.001536,0.359674,away_goals
1,home_defense,0.128500,0.183187,0.128500,away_goals
2,away_attack,0.117326,0.196387,0.117326,away_goals
3,form_diff,0.097827,0.467208,0.097827,away_goals
4,defense_diff,-0.089568,-0.106527,0.089568,away_goals
5,away_form,-0.077091,-0.483010,0.077091,away_goals
6,home_matches_played,-0.055997,-0.000241,0.055997,away_goals
7,home_attack,0.055114,0.091305,0.055114,away_goals
8,attack_diff,-0.051705,-0.072600,0.051705,away_goals
9,home_form,0.051103,0.319284,0.051103,away_goals


In [12]:
def calculate_permutation_importance(model, features_data, target, model_name):
    result = permutation_importance(
        model,
        features_data[path_a.FEATURE_COLUMNS],
        target,
        n_repeats=10,
        random_state=42,
        scoring="neg_mean_absolute_error",
    )

    importance_data = pd.DataFrame(
        {
            "feature": path_a.FEATURE_COLUMNS,
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std,
            "model": model_name,
        }
    )

    return importance_data.sort_values(
        "importance_mean",
        ascending=False,
    ).reset_index(drop=True)


home_importance = calculate_permutation_importance(
    home_model,
    features_data,
    home_goals,
    "home_goals",
)

away_importance = calculate_permutation_importance(
    away_model,
    features_data,
    away_goals,
    "away_goals",
)

display(home_importance.head(30))
display(away_importance.head(30))

,feature,importance_mean,importance_std,model
0,elo_diff,0.201218,0.001297,home_goals
1,form_diff,0.039493,0.001065,home_goals
2,away_defense,0.024965,0.000771,home_goals
3,away_form,0.022890,0.000801,home_goals
4,home_attack,0.013188,0.000519,home_goals
5,away_matches_played,0.010738,0.000938,home_goals
6,home_form,0.009877,0.000500,home_goals
7,defense_diff,0.009498,0.000494,home_goals
8,fifa_points_diff,0.004633,0.000229,home_goals
9,attack_diff,0.002875,0.000548,home_goals


,feature,importance_mean,importance_std,model
0,elo_diff,0.116540,0.002086,away_goals
1,form_diff,0.022810,0.000524,away_goals
2,home_defense,0.016522,0.000638,away_goals
3,away_form,0.011200,0.000377,away_goals
4,away_attack,0.009800,0.000624,away_goals
5,home_form,0.007121,0.000326,away_goals
6,home_attack,0.005620,0.000388,away_goals
7,defense_diff,0.004391,0.000438,away_goals
8,fifa_points_diff,0.002605,0.000132,away_goals
9,neutral,0.002340,0.000354,away_goals


In [13]:
simulation_results = path_a.run_simulations(
    schedule_data=schedule_data,
    playoff_data=None,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    simulations=100_000,
    seed=42,
)

In [14]:
simulation_results

{'champion': Counter({'Spain': 22601,
          'Argentina': 10806,
          'Brazil': 7865,
          'France': 7571,
          'England': 5487,
          'Germany': 5455,
          'Colombia': 3786,
          'Netherlands': 3733,
          'Portugal': 3434,
          'Switzerland': 3268,
          'Belgium': 2846,
          'Mexico': 2806,
          'Norway': 2628,
          'Uruguay': 2442,
          'Japan': 2008,
          'Ecuador': 1840,
          'Morocco': 1599,
          'South Korea': 1285,
          'United States': 1051,
          'Turkey': 932,
          'Austria': 876,
          'Paraguay': 737,
          'Canada': 701,
          'Scotland': 574,
          'Croatia': 515,
          'Sweden': 442,
          'Iran': 436,
          'Australia': 409,
          'Senegal': 381,
          'Czechia': 365,
          'Algeria': 286,
          'Egypt': 180,
          'Ivory Coast': 142,
          'Panama': 100,
          'Uzbekistan': 73,
          'Jordan': 67,
          'Tunisia

In [15]:
def save_conditional_playoff_champion_probabilities_safe(
    champion_given_playoff_counter,
    playoff_qualification_counter,
    output_path,
):
    columns = [
        "team",
        "playoff_qualification_count",
        "champion_count",
        "champion_probability_given_qualification",
    ]

    if not playoff_qualification_counter:
        pd.DataFrame(columns=columns).to_csv(output_path, index=False)
        return

    rows = []

    for team, qualification_count in playoff_qualification_counter.items():
        champion_count = champion_given_playoff_counter.get(team, 0)

        rows.append(
            {
                "team": team,
                "playoff_qualification_count": qualification_count,
                "champion_count": champion_count,
                "champion_probability_given_qualification": (
                    champion_count / qualification_count
                    if qualification_count > 0
                    else 0.0
                ),
            }
        )

    probabilities_data = pd.DataFrame(rows, columns=columns).sort_values(
        "champion_probability_given_qualification",
        ascending=False,
    )

    probabilities_data.to_csv(output_path, index=False)


path_a.save_conditional_playoff_champion_probabilities = (
    save_conditional_playoff_champion_probabilities_safe
)

In [16]:
def save_counter_probabilities_safe(
    counter,
    simulations,
    output_path,
    probability_column,
):
    rows = [
        {
            "team": team,
            "count": count,
            probability_column: count / simulations,
        }
        for team, count in counter.most_common()
    ]

    probabilities_data = pd.DataFrame(
        rows,
        columns=["team", "count", probability_column],
    )

    probabilities_data.to_csv(output_path, index=False)

In [17]:
def save_outputs_with_round_tracking(results, output_dir, seed):
    output_dir.mkdir(parents=True, exist_ok=True)

    simulations = int(results["simulations"])

    output_specs = {
        "champion": ("champion_probabilities.csv", "champion_probability"),
        "runner_up": ("runner_up_probabilities.csv", "runner_up_probability"),
        "round_of_32": ("round_of_32_probabilities.csv", "round_of_32_probability"),
    }

    for result_key, (filename, probability_column) in output_specs.items():
        save_counter_probabilities_safe(
            counter=results[result_key],
            simulations=simulations,
            output_path=output_dir / filename,
            probability_column=probability_column,
        )

    metadata = {
        "model": "path_a_poisson_dynamic_ratings_with_round_tracking",
        "simulations": simulations,
        "seed": seed,
        "rounds_saved": list(output_specs.keys()),
    }

    (output_dir / "run_metadata.json").write_text(
        path_a.json.dumps(metadata, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

In [18]:
save_outputs_with_round_tracking(
    results=simulation_results,
    output_dir=output_dir,
    seed=42,
)